## Imports

In [1]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Setup

In [2]:
MODEL_NAME = "google-bert/bert-base-uncased"
SEED = 42
NUM_LABELS = 3

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Load MNLI Dataset

In [3]:
mnli = load_dataset("nyu-mll/glue", "mnli")

train_dataset = mnli["train"]
val_matched = mnli["validation_matched"]
val_mismatched = mnli["validation_mismatched"]

print("Train:", len(train_dataset))
print("Validation matched:", len(val_matched))
print("Validation mismatched:", len(val_mismatched))

Train: 392702
Validation matched: 9815
Validation mismatched: 9832


In [4]:
mnli

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

## BERT Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Sample

In [6]:
example = train_dataset[0]

print("Premise:")
print(example["premise"])

print("\nHypothesis:")
print(example["hypothesis"])

Premise:
Conceptually cream skimming has two basic dimensions - product and geography.

Hypothesis:
Product and geography are what make cream skimming work. 


In [7]:
encoded = tokenizer(
    example["premise"],
    example["hypothesis"],
    truncation=True
)

encoded

{'input_ids': [101, 17158, 2135, 6949, 8301, 25057, 2038, 2048, 3937, 9646, 1011, 4031, 1998, 10505, 1012, 102, 4031, 1998, 10505, 2024, 2054, 2191, 6949, 8301, 25057, 2147, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
tokens

['[CLS]',
 'conceptual',
 '##ly',
 'cream',
 'ski',
 '##mming',
 'has',
 'two',
 'basic',
 'dimensions',
 '-',
 'product',
 'and',
 'geography',
 '.',
 '[SEP]',
 'product',
 'and',
 'geography',
 'are',
 'what',
 'make',
 'cream',
 'ski',
 '##mming',
 'work',
 '.',
 '[SEP]']

## Tokenization Function

In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

## Tokenize Dataset

In [16]:
tokenized_mnli = mnli.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

Map:   0%|          | 0/9796 [00:00<?, ? examples/s]

Map:   0%|          | 0/9847 [00:00<?, ? examples/s]

In [17]:
tokenized_mnli

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9847
    })
})

In [18]:
tokenized_mnli["train"][0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1,
 'idx': 0,
 'input_ids': [101,
  17158,
  2135,
  6949,
  8301,
  25057,
  2038,
  2048,
  3937,
  9646,
  1011,
  4031,
  1998,
  10505,
  1012,
  102,
  4031,
  1998,
  10505,
  2024,
  2054,
  2191,
  6949,
  8301,
  25057,
  2147,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [19]:
print("Premise:")
print(tokenized_mnli["train"][0]["premise"])

print("\nHypothesis:")
print(tokenized_mnli["train"][0]["hypothesis"])

print("\nLabel:")
print(tokenized_mnli["train"][0]["label"])

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        tokenized_mnli["train"][0]["input_ids"]
    )
)

Premise:
Conceptually cream skimming has two basic dimensions - product and geography.

Hypothesis:
Product and geography are what make cream skimming work. 

Label:
1

Tokens:
['[CLS]', 'conceptual', '##ly', 'cream', 'ski', '##mming', 'has', 'two', 'basic', 'dimensions', '-', 'product', 'and', 'geography', '.', '[SEP]', 'product', 'and', 'geography', 'are', 'what', 'make', 'cream', 'ski', '##mming', 'work', '.', '[SEP]']


### Checking length

In [20]:
lengths = [
    len(
        tokenizer(
            premise,
            hypothesis,
            truncation=False
        )["input_ids"]
    )
    for premise, hypothesis in zip(
        mnli["train"]["premise"],
        mnli["train"]["hypothesis"]
    )
]

num_over_128 = sum(length > 128 for length in lengths)

print(f"Examples > 128 tokens: {num_over_128:,}")
print(f"Percentage truncated: {num_over_128 / len(lengths) * 100:.2f}%")
print(f"Longest example: {max(lengths)} tokens")

Examples > 128 tokens: 1,169
Percentage truncated: 0.30%
Longest example: 444 tokens
